# MaIA_SDCI Temperature Control

**Sistemas de Decision y Control Inteligente (SDCI)**  
**Universidad de los Andes, 2026**

Copyright (c) 2026 Leffer Trochez

---

Equation-based simulation model for the Python project.

This notebook contains the plant, actuator, sensor, disturbance, and simulation execution logic. It intentionally contains **no controller algorithm**.

The model communicates with the selected controller only through:

`duty_cycle = controller.step(T_ref, T_error, T_meas)`

All controller implementations are isolated in `config/controller/`.


In [ ]:
from __future__ import annotations

import numpy as np


class MaIA_SDCI_Temperature_Control:
    """Complete equation-based model used by the Python project."""

    CONTROLLER_NAMES = {
        1: "ON_OFF",
        2: "FUZZY",
        3: "MPC",
        4: "ESC",
        5: "REPLICATOR",
    }

    def __init__(self, CFG, MODEL, ENV, SENSOR, CTRL, controller_id=1):
        self.CFG = CFG
        self.MODEL = MODEL
        self.ENV = ENV
        self.SENSOR = SENSOR
        self.CTRL = CTRL
        self.controller_id = int(controller_id)

        if self.controller_id not in self.CONTROLLER_NAMES:
            raise ValueError("Controller ID must be an integer from 1 to 5.")

        self.integration_step = 0.1
        self.reset()

    def _create_selected_controller(self):
        """Create the selected controller through the common project registry."""
        factory = self.CTRL.get("createController")
        if not callable(factory):
            raise ControllerInterfaceError(
                "Controller registry is incomplete. "
                "Expected CTRL['createController'] to be callable."
            )

        controller = factory(self.controller_id, self.CTRL)
        controller.reset()
        return controller

    def reset(self):
        self.time = 0.0
        self.plant_four = np.asarray(self.MODEL["PLANT"]["initialTemperature"], dtype=float).copy()
        self.plant_single = float(self.MODEL["PLANT"]["initialTemperatureSingleRoom"])
        self.sensor_state = np.asarray(self.SENSOR["initialTemperature"], dtype=float).copy()
        self.random_generators = [
            np.random.default_rng(int(seed)) for seed in self.SENSOR["seed"]
        ]

        self.controller = self._create_selected_controller()

        self.T_meas = self._sample_sensors()
        self.T_ref = self.reference_vector()
        self.T_error = self.T_ref - self.T_meas
        self.duty_cycle = self._controller_output()
        self.Q_heat = self._actuator_output(self.duty_cycle)

    def set_controller(self, controller_id):
        self.controller_id = int(controller_id)
        if self.controller_id not in self.CONTROLLER_NAMES:
            raise ValueError("Controller ID must be an integer from 1 to 5.")
        self.reset()

    def reference_vector(self):
        if self.controller_id == 4:
            return np.array(
                [self.CFG["reference"]["singleRoom"], 0.0, 0.0, 0.0], dtype=float
            )
        return np.asarray(self.CFG["reference"]["fourRooms"], dtype=float).copy()

    def disturbance_vector(self, time_seconds=None):
        if time_seconds is None:
            time_seconds = self.time
        disturbance = self.ENV["disturbance"]
        if not disturbance["enabled"]:
            return np.zeros(4, dtype=float)
        start = 60.0 * float(disturbance["startTime"])
        end = 60.0 * float(disturbance["endTime"])
        if start <= time_seconds < end:
            return float(disturbance["amplitude"]) * np.asarray(
                disturbance["roomMask"], dtype=float
            )
        return np.zeros(4, dtype=float)

    def _controller_output(self):
        """
        Execute the selected controller through the fixed interface.

        Inputs:
            T_ref, T_error, T_meas

        Output:
            duty_cycle
        """
        self.T_error = self.T_ref - self.T_meas

        T_ref = np.asarray(self.T_ref, dtype=float).reshape(4)
        T_error = np.asarray(self.T_error, dtype=float).reshape(4)
        T_meas = np.asarray(self.T_meas, dtype=float).reshape(4)

        try:
            duty_cycle = self.controller.step(
                T_ref.copy(),
                T_error.copy(),
                T_meas.copy(),
            )
        except ControllerInterfaceError:
            raise
        except NotImplementedError as exc:
            raise ControllerInterfaceError(
                f"{self.CONTROLLER_NAMES[self.controller_id]} "
                "controller is not implemented."
            ) from exc
        except Exception as exc:
            raise ControllerInterfaceError(
                f"{self.CONTROLLER_NAMES[self.controller_id]} "
                f"controller execution failed: {exc}"
            ) from exc

        try:
            duty_cycle = np.asarray(duty_cycle, dtype=float).reshape(-1)
        except Exception as exc:
            raise ControllerInterfaceError(
                "Controller output duty_cycle must be convertible "
                "to a numeric vector."
            ) from exc

        if duty_cycle.size != 4:
            raise ControllerInterfaceError(
                "Controller output duty_cycle must contain exactly four elements."
            )

        if not np.all(np.isfinite(duty_cycle)):
            raise ControllerInterfaceError(
                "Controller output duty_cycle must contain only finite values."
            )

        # Final interface protection. Controllers should already respect [0, 1].
        return np.clip(duty_cycle, 0.0, 1.0)

    def _actuator_output(self, duty_cycle):
        cfg = self.MODEL["ACTUATOR"]
        duty = np.clip(np.asarray(duty_cycle, dtype=float), 0.0, 1.0)
        cycles = np.floor(duty * cfg["cyclesPerWindow"] + 0.5)
        power_per_cycle = (
            cfg["P_rated"] * cfg["efficiency"] / cfg["cyclesPerWindow"]
        )
        return cycles * power_per_cycle

    def _plant_output_vector(self):
        if self.controller_id == 4:
            return np.array([self.plant_single, 0.0, 0.0, 0.0], dtype=float)
        return self.plant_four.copy()

    def _sample_sensors(self):
        measurement = self.sensor_state.copy()
        for room in range(4):
            measurement[room] += float(self.SENSOR["bias"][room])
            measurement[room] += self.random_generators[room].normal(
                0.0, float(self.SENSOR["noiseStd"][room])
            )
        if self.controller_id == 4:
            measurement[1:] = 0.0
        return measurement

    def _combined_derivative(self, plant_state, sensor_state, heat_power, ambient_vector):
        thermal_capacitance = float(self.MODEL["PLANT"]["C"])
        alpha = float(self.MODEL["PLANT"]["alpha"])
        sensor_tau = float(self.SENSOR["tau"])

        if self.controller_id == 4:
            temperature = float(plant_state[0])
            temperature_derivative = (
                heat_power[0] + alpha * (ambient_vector[0] - temperature)
            ) / thermal_capacitance
            room_vector = np.array([temperature, 0.0, 0.0, 0.0], dtype=float)
            sensor_derivative = (room_vector - sensor_state) / sensor_tau
            return np.array([temperature_derivative], dtype=float), sensor_derivative

        temperature = np.asarray(plant_state, dtype=float)
        derivative = np.zeros(4, dtype=float)
        derivative[0] = (
            heat_power[0]
            + alpha * (ambient_vector[0] - temperature[0])
            + alpha * (temperature[1] - temperature[0])
        ) / thermal_capacitance
        derivative[1] = (
            heat_power[1]
            + alpha * (ambient_vector[1] - temperature[1])
            + alpha * (temperature[0] - temperature[1])
            + alpha * (temperature[2] - temperature[1])
        ) / thermal_capacitance
        derivative[2] = (
            heat_power[2]
            + alpha * (ambient_vector[2] - temperature[2])
            + alpha * (temperature[1] - temperature[2])
            + alpha * (temperature[3] - temperature[2])
        ) / thermal_capacitance
        derivative[3] = (
            heat_power[3]
            + alpha * (ambient_vector[3] - temperature[3])
            + alpha * (temperature[2] - temperature[3])
        ) / thermal_capacitance

        sensor_derivative = (temperature - sensor_state) / sensor_tau
        return derivative, sensor_derivative

    def _integrate_one_second(self):
        remaining = 1.0
        while remaining > 1e-12:
            step = min(self.integration_step, remaining)
            current_time = self.time + (1.0 - remaining)
            disturbance = self.disturbance_vector(current_time)
            ambient = float(self.ENV["T_a"]) + disturbance

            if self.controller_id == 4:
                plant = np.array([self.plant_single], dtype=float)
            else:
                plant = self.plant_four.copy()
            sensor = self.sensor_state.copy()

            k1_plant, k1_sensor = self._combined_derivative(
                plant, sensor, self.Q_heat, ambient
            )
            k2_plant, k2_sensor = self._combined_derivative(
                plant + 0.5 * step * k1_plant,
                sensor + 0.5 * step * k1_sensor,
                self.Q_heat,
                ambient,
            )
            k3_plant, k3_sensor = self._combined_derivative(
                plant + 0.5 * step * k2_plant,
                sensor + 0.5 * step * k2_sensor,
                self.Q_heat,
                ambient,
            )
            k4_plant, k4_sensor = self._combined_derivative(
                plant + step * k3_plant,
                sensor + step * k3_sensor,
                self.Q_heat,
                ambient,
            )

            next_plant = plant + step * (
                k1_plant + 2.0 * k2_plant + 2.0 * k3_plant + k4_plant
            ) / 6.0
            next_sensor = sensor + step * (
                k1_sensor + 2.0 * k2_sensor + 2.0 * k3_sensor + k4_sensor
            ) / 6.0

            if self.controller_id == 4:
                self.plant_single = float(next_plant[0])
            else:
                self.plant_four = next_plant
            self.sensor_state = next_sensor
            remaining -= step

    def step(self):
        """Advance the model by one simulation second."""
        self._integrate_one_second()
        self.time += 1.0
        self.T_meas = self._sample_sensors()
        self.T_ref = self.reference_vector()
        self.T_error = self.T_ref - self.T_meas
        self.duty_cycle = self._controller_output()
        self.Q_heat = self._actuator_output(self.duty_cycle)
        return self.current_sample()

    def current_sample(self):
        return {
            "time": float(self.time),
            "T_ref": self.T_ref.copy(),
            "T_error": self.T_error.copy(),
            "T_meas": self.T_meas.copy(),
            "Q_heat": self.Q_heat.copy(),
            "duty_cycle": self.duty_cycle.copy(),
            "T_room": self._plant_output_vector(),
            "T_dist": self.disturbance_vector(),
        }

    def run(self, stop_time=None):
        """Run a complete simulation without GUI pacing."""
        if stop_time is None:
            stop_time = float(self.CFG["simulation"]["stopTime"])
        stop_time = float(stop_time)

        samples = [self.current_sample()]
        while self.time < stop_time - 1e-12:
            samples.append(self.step())

        time = np.array([sample["time"] for sample in samples], dtype=float)
        return {
            "time": time,
            "T_ref": np.vstack([sample["T_ref"] for sample in samples]),
            "T_error": np.vstack([sample["T_error"] for sample in samples]),
            "T_meas": np.vstack([sample["T_meas"] for sample in samples]),
            "Q_heat": np.vstack([sample["Q_heat"] for sample in samples]),
            "duty_cycle": np.vstack([sample["duty_cycle"] for sample in samples]),
            "T_room": np.vstack([sample["T_room"] for sample in samples]),
            "T_dist": np.vstack([sample["T_dist"] for sample in samples]),
        }
